In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn.model_selection as skms
import sklearn.preprocessing as skpre
import sklearn.metrics as skmet

import sklearn.linear_model as sklin
import sklearn.ensemble as ske
import sklearn.naive_bayes as sknb
import sklearn.svm as sksvm
import sklearn.neighbors as skn

In [2]:
# Set variables
missing_data_strat = 'impute'     # drop | impute
scale_features = True   # True | False
data_file_path = '../data/titanic/'

Read in datasets and do EDA (not shown) to understand how features need to be preprocessed

In [3]:
dataset_train_raw = pd.read_csv(data_file_path+'train.csv')
dataset_test_raw = pd.read_csv(data_file_path+'test.csv')

# PassengerId needed for output, not training
# Name - Drop
# Sex - One hot encode
# Age - Decide whether to drop or impute missing values (prob impute)
# Ticket - Look at examples and decide whether any useful info can be extracted; otherwise, drop column
# Cabin - Look at examples and decide whether any useful info can be extracted; otherwise, drop column
# Embarked - One hot encode + handle 2 missing values

dataset_test_raw.describe(include='all')

# After EDA...
    # Impute Age
    # Drop Ticket
    # Convert Cabin number to HasCabin variable

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
count,418.000000,418.000000,418,418,332.000000,418.000000,418.000000,418,417.000000,91,418
unique,NaN,NaN,418,2,NaN,NaN,NaN,363,NaN,76,3
top,NaN,NaN,"Kelly, Mr. James",male,NaN,NaN,NaN,PC 17608,NaN,B57 B59 B63 B66,S
freq,NaN,NaN,1,266,NaN,NaN,NaN,5,NaN,3,270
mean,1100.500000,2.265550,NaN,NaN,30.272590,0.447368,0.392344,NaN,35.627188,NaN,NaN
std,120.810458,0.841838,NaN,NaN,14.181209,0.896760,0.981429,NaN,55.907576,NaN,NaN
min,892.000000,1.000000,NaN,NaN,0.170000,0.000000,0.000000,NaN,0.000000,NaN,NaN
25%,996.250000,1.000000,NaN,NaN,21.000000,0.000000,0.000000,NaN,7.895800,NaN,NaN
50%,1100.500000,3.000000,NaN,NaN,27.000000,0.000000,0.000000,NaN,14.454200,NaN,NaN
75%,1204.750000,3.000000,NaN,NaN,39.000000,1.000000,0.000000,NaN,31.500000,NaN,NaN


Pre-process training and test datasets

In [4]:
# Drop Ticket, Name, PassengerId
dataset_train_cleaned = dataset_train_raw.drop(['Ticket', 'Name', 'PassengerId'], axis = 1)
dataset_test_cleaned = dataset_test_raw.drop(['Ticket', 'Name', 'PassengerId'], axis = 1)

# Convert Cabin to HasCabin based on whether a value exists
dataset_train_cleaned['HasCabin'] = dataset_train_cleaned['Cabin'].notna().astype(int)
dataset_train_cleaned.drop(['Cabin'], axis = 1, inplace=True)

dataset_test_cleaned['HasCabin'] = dataset_test_cleaned['Cabin'].notna().astype(int)
dataset_test_cleaned.drop(['Cabin'], axis = 1, inplace=True)

# Impute missing Age, Embarked, and Fare datapoints (Fare only missing in test)
dataset_train_cleaned['Age'] = dataset_train_cleaned['Age'].fillna(dataset_train_cleaned['Age'].median())
dataset_train_cleaned['Embarked'] = dataset_train_cleaned['Embarked'].fillna(dataset_train_cleaned['Embarked'].mode()[0])
dataset_train_cleaned['Fare'] = dataset_train_cleaned['Age'].fillna(dataset_train_cleaned['Age'].median()) # For clarity only; should do nothing

dataset_test_cleaned['Age'] = dataset_test_cleaned['Age'].fillna(dataset_train_cleaned['Age'].median())   # Fill with median of *TRAIN data
dataset_test_cleaned['Embarked'] = dataset_test_cleaned['Embarked'].fillna(dataset_train_cleaned['Embarked'].mode()[0])    # Fill with mode of *TRAIN data
dataset_test_cleaned['Fare'] = dataset_train_cleaned['Age'].fillna(dataset_train_cleaned['Age'].median())

# One hot encode Sex and Embarked
dataset_train_cleaned = pd.get_dummies(dataset_train_cleaned, columns=['Sex', 'Embarked'], drop_first=True, dtype=int)
dataset_test_cleaned = pd.get_dummies(dataset_test_cleaned, columns=['Sex', 'Embarked'], drop_first=True, dtype=int)

print(dataset_train_cleaned.describe(include='all'))
print(dataset_test_cleaned.describe(include='all'))

         Survived      Pclass         Age       SibSp       Parch        Fare  \
count  891.000000  891.000000  891.000000  891.000000  891.000000  891.000000   
mean     0.383838    2.308642   29.361582    0.523008    0.381594   29.361582   
std      0.486592    0.836071   13.019697    1.102743    0.806057   13.019697   
min      0.000000    1.000000    0.420000    0.000000    0.000000    0.420000   
25%      0.000000    2.000000   22.000000    0.000000    0.000000   22.000000   
50%      0.000000    3.000000   28.000000    0.000000    0.000000   28.000000   
75%      1.000000    3.000000   35.000000    1.000000    0.000000   35.000000   
max      1.000000    3.000000   80.000000    8.000000    6.000000   80.000000   

         HasCabin    Sex_male  Embarked_Q  Embarked_S  
count  891.000000  891.000000  891.000000  891.000000  
mean     0.228956    0.647587    0.086420    0.725028  
std      0.420397    0.477990    0.281141    0.446751  
min      0.000000    0.000000    0.000000    0

In [5]:
X_train = dataset_train_cleaned.loc[:, dataset_train_cleaned.columns != 'Survived']
y_train = dataset_train_cleaned.loc[:, dataset_train_cleaned.columns == 'Survived']
X_test = dataset_test_cleaned.loc[:, dataset_test_cleaned.columns != 'Survived']    # Note: test dataset does not include a target var; explicitly excluding just for clarity

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)

(891, 9)
(891, 1)
(418, 9)


Scale features

In [6]:
if (scale_features):
    scalar = skpre.StandardScaler()
    X_train = scalar.fit_transform(X_train)
    X_test = scalar.transform(X_test)   # Do not refit; use same scaling as training data

Predict outcomes

In [7]:
# classifier = sklin.LogisticRegression()
# classifier = ske.RandomForestClassifier(n_estimators=100)
# classifier = sknb.GaussianNB()
classifier = sksvm.SVC(kernel='rbf') ####
# classifier = sksvm.SVC(kernel='linear')
# classifier = sksvm.SVC(kernel='poly', degree=2)
# classifier = sksvm.SVC(kernel='poly', degree=3)
# classifier = skn.KNeighborsClassifier(n_neighbors=5)
# classifier = skn.KNeighborsClassifier(n_neighbors=15)


classifier.fit(X_train, y_train)

y_pred = classifier.predict(X_test)

print(y_pred)

[0 0 0 0 0 0 1 0 1 0 0 0 1 0 1 1 0 0 0 1 0 0 1 0 1 0 1 0 0 0 0 0 0 0 1 0 0
 0 0 0 0 0 0 1 1 0 1 0 1 0 0 0 1 1 0 0 0 0 0 1 0 0 0 1 1 1 1 0 1 0 1 0 0 1
 1 1 0 1 0 1 0 0 0 0 0 0 1 0 1 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 0 0 0
 1 1 1 1 0 0 1 1 1 1 0 1 0 0 1 0 1 0 0 0 1 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0
 0 0 1 0 0 0 0 0 1 0 0 0 1 0 1 0 0 0 1 0 1 0 0 0 0 0 0 1 1 0 1 1 0 1 1 0 1
 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 0 0 1 0 1 0 0 0 0 1 0 0 1 0 1 0 1 0
 1 0 1 1 0 1 0 0 0 1 0 0 1 0 0 0 1 1 1 1 1 0 0 0 1 0 1 0 1 0 1 0 0 0 0 0 1
 0 0 0 1 0 0 0 0 0 0 0 1 0 1 1 0 1 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 1 0 0 0 0
 1 0 0 0 0 1 0 0 1 1 0 1 0 0 0 0 0 1 1 1 1 0 0 0 0 0 0 0 1 0 1 0 0 0 1 1 0
 1 0 0 0 0 0 0 0 0 0 1 0 1 0 1 0 1 1 0 0 0 0 0 1 0 0 0 0 1 1 0 1 0 0 0 1 0
 0 1 0 0 1 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 1 0 1 0 0 1 0 1 0 1 1 0
 1 1 0 1 1 0 0 1 0 0 0]


c:\Users\madam\anaconda3\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Sanity checking

In [8]:
# Create new test dataset based only on training data with "source of truth"
X_train_sc, X_test_sc, y_train_sc, y_test_sc = skms.train_test_split(X_train, y_train, test_size=0.25)

y_pred_sc = classifier.predict(X_test_sc)

In [9]:
# Sanity check - Confusion matrix for training data

print(skmet.confusion_matrix(y_pred = y_pred_sc, y_true=y_test_sc))
print(skmet.accuracy_score(y_pred = y_pred_sc, y_true=y_test_sc))

[[124   9]
 [ 26  64]]
0.8430493273542601


Final Output

In [10]:
# Reformat predictions for submission
ids = dataset_test_raw['PassengerId']
predictions = y_pred
combined_data = np.stack((ids, predictions), axis=1)

submission = pd.DataFrame(combined_data, columns=['PassengerId', 'Survived'])

pd.set_option('display.max_rows', None)
print("Predicted %Survived: {:.1f}%".format(submission['Survived'].mean()*100))
print(submission)

submission.to_csv('titanic_submission.csv', index=False)

Predicted %Survived: 32.5%
     PassengerId  Survived
0            892         0
1            893         0
2            894         0
3            895         0
4            896         0
5            897         0
6            898         1
7            899         0
8            900         1
9            901         0
10           902         0
11           903         0
12           904         1
13           905         0
14           906         1
15           907         1
16           908         0
17           909         0
18           910         0
19           911         1
20           912         0
21           913         0
22           914         1
23           915         0
24           916         1
25           917         0
26           918         1
27           919         0
28           920         0
29           921         0
30           922         0
31           923         0
32           924         0
33           925         0
34           926         1
3